# An unfitted time-dependant Stokes problem

We can solve the Stokes problem on a moving (unfitted) domain. Here we show an example of a rising ball that the fluid moves around.

In [ ]:
from ngsolve import *
from xfem import *
import ngsolve.webgui as ngw
from netgen.occ import *

from ngsxditto.transport import *
from ngsxditto.levelset import *
from ngsxditto.fluid import *
from ngsxditto.redistancing import *
from ngsxditto.gradient_tester import *

In [ ]:
domain = MoveTo(-1, -1).Rectangle(2, 2).Face()
domain.edges.Max(X).name = "right"
domain.edges.Min(X).name = "left"
domain.edges.Min(Y).name = "bottom"
domain.edges.Max(Y).name = "top"
mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=0.1))

In [ ]:
dt = 0.02
t = Parameter(0)
starting_levelset = -((x**2 + (y+0.5-t)**2)**(1/2) - 0.5)
transport = KnownSolutionTransport(mesh, starting_levelset, dt=dt, time=t)
levelset = LevelSetGeometry(transport)
levelset.Initialize(starting_levelset)

In [ ]:
fluid_params = FluidParameters(viscosity=1e-3)
fluid = TaylorHood(mesh, fluid_params, lset=levelset, dt=dt)
fluid.SetBoundaryConditions(dirichlet={"left": CF(((1-y)*(1+y), 0)), "top": CF((0, 0)), "bottom": CF((0, 0))})
fluid.InitializeSpaces(dbnd="left|top|bottom")
fluid.UpdateActiveDofs()
fluid.InitializeForms()
sol = fluid.SolveStokes()
fluid.SetInitialValues(sol.components[0], sol.components[1])

gfvis = GridFunction(fluid.V, multidim=0)
gfvis_tmp = GridFunction(fluid.V)

t.Set(0)
while t < 1:
    levelset.OneStep()
    fluid.OneStep()

    gfvis_tmp.Set(IfPos(levelset.lsetp1, CF((0, 0)), fluid.gfu.components[0]))
    gfvis.AddMultiDimComponent(gfvis_tmp.vec)

ngw.Draw(gfvis, mesh, interpolate_multidim=True,animate=True, deformation=levelset.deformation)
